# Initial settings

In [ ]:
DRIVE_PROJECT = "/content/drive/MyDrive/si_masterthesis_colab"

INPUT_ZIPS = [
    "mock_rho-0.5_seed1.zip",
    "mock_rho-0.5_seed2.zip",
    "mock_rho0_seed1.zip",
    "mock_rho0_seed2.zip",
    "mock_rho1_seed1.zip",
    "mock_rho1_seed2.zip",
    "mock_rho1_seed3.zip",
]

RESULT_FOLDER_NAME = "production_all"

WEIGHT = "boundary_length_over_distance"

# Mount to Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Check Zips

In [ ]:
from pathlib import Path

drive_project = Path(DRIVE_PROJECT)
input_dir = drive_project / "input_data"

missing = []

for zip_name in INPUT_ZIPS:
    path = input_dir / zip_name

    if path.exists():
        size_gb = path.stat().st_size / 1024**3
        print(f"OK: {zip_name} ({size_gb:.2f} GB)")
    else:
        print(f"不足: {zip_name}")
        missing.append(path)

if missing:
    raise FileNotFoundError(
        "以下のZIPがありません:\n"
        + "\n".join(str(path) for path in missing)
    )

# Take codes from GitHub

In [ ]:
import shutil
from pathlib import Path

repo_dir = Path("/content/pipeline")

if repo_dir.exists():
    shutil.rmtree(repo_dir)

!git clone -q https://github.com/Sorao0921/biaxial-cpfem.git /content/pipeline

script = repo_dir / "tools/theme1/analyze_graph_spectra.py"

print("解析スクリプト:", script)
print("存在:", script.exists())

# Install libraries

In [ ]:
%cd /content/pipeline

!pip install -q -e .

# Check Strage in Colab

In [ ]:
import shutil

total, used, free = shutil.disk_usage("/content")

print(f"全容量: {total / 1024**3:.1f} GB")
print(f"使用済み: {used / 1024**3:.1f} GB")
print(f"空き容量: {free / 1024**3:.1f} GB")

# Extract Zips

In [ ]:
import zipfile
import shutil
from pathlib import Path

repo_dir = Path("/content/pipeline")
input_dir = Path(DRIVE_PROJECT) / "input_data"
temporary_extract = Path("/content/input_extract")

if temporary_extract.exists():
    shutil.rmtree(temporary_extract)

temporary_extract.mkdir(parents=True)

for zip_name in INPUT_ZIPS:
    zip_path = input_dir / zip_name

    print(f"\n展開中: {zip_name}")

    # ZIPごとに一時展開先を空にする
    for child in temporary_extract.iterdir():
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()

    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(temporary_extract)

    # ZIP内のdatabaseとoutputsを自動探索
    database_candidates = [
        path for path in temporary_extract.rglob("database")
        if path.is_dir()
    ]

    outputs_candidates = [
        path for path in temporary_extract.rglob("outputs")
        if path.is_dir()
    ]

    if len(database_candidates) != 1:
        raise RuntimeError(
            f"{zip_name}: databaseフォルダを一意に検出できません。\n"
            + "\n".join(str(path) for path in database_candidates)
        )

    if len(outputs_candidates) != 1:
        raise RuntimeError(
            f"{zip_name}: outputsフォルダを一意に検出できません。\n"
            + "\n".join(str(path) for path in outputs_candidates)
        )

    shutil.copytree(
        database_candidates[0],
        repo_dir / "database",
        dirs_exist_ok=True,
    )

    shutil.copytree(
        outputs_candidates[0],
        repo_dir / "outputs",
        dirs_exist_ok=True,
    )

    print("統合完了")

shutil.rmtree(temporary_extract)

print("\nすべての入力ZIPを統合しました。")

# Check files

In [ ]:
%cd /content/pipeline

from pathlib import Path
from collections import Counter

from src.dashboard.catalog import scan_outputs
from src.theme1.contribution import complete_cases

outputs_dir = Path("/content/pipeline/outputs")
spatial_dir = Path("/content/pipeline/database/spatial_model")

records = scan_outputs(
    outputs_dir,
    prefer_raw_height=True,
)

cases = complete_cases(records, spatial_dir)

print("検出された完全なケース数:", len(cases))

print("\n種類別ファイル数:")
record_counts = Counter(record.kind for record in records)

for kind, count in sorted(record_counts.items()):
    print(f"  {kind}: {count}")

print("\nrho × seed 別ケース数:")
batch_counts = Counter(
    (case.rho, case.seed)
    for case in cases
)

for (rho, seed), count in sorted(batch_counts.items()):
    print(f"  rho={rho:g}, seed={seed}: {count}")

print("\ntexture別ケース数:")
texture_counts = Counter(case.texture for case in cases)

for texture, count in sorted(texture_counts.items()):
    print(f"  {texture}: {count}")

print("\nstate別ケース数:")
state_counts = Counter(case.state for case in cases)

for state, count in sorted(state_counts.items()):
    print(f"  state{state:02d}: {count}")

if not cases:
    raise RuntimeError(
        "完全なケースを1件も検出できませんでした。"
    )

# Case check

In [ ]:
import pandas as pd

case_table = pd.DataFrame([
    {
        "case_id": case.case_id,
        "rho": case.rho,
        "seed": case.seed,
        "texture": case.texture,
        "sd": case.sd,
        "state": case.state,
    }
    for case in cases
])

display(case_table.head(20))

print("rho:", sorted(case_table["rho"].unique()))
print("seed:", sorted(case_table["seed"].unique()))
print("texture:", sorted(case_table["texture"].unique()))
print("sd:", sorted(case_table["sd"].unique()))
print("state:", sorted(case_table["state"].unique()))

In [ ]:
manifest_path = (
    Path(DRIVE_PROJECT)
    / "results"
    / RESULT_FOLDER_NAME
    / "input_case_manifest.csv"
)

manifest_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

case_table.to_csv(
    manifest_path,
    index=False,
)

print("ケース一覧を保存:", manifest_path)

# Run analysis

In [ ]:
%cd /content/pipeline

OUTPUT_DIR = "/content/pipeline/database/theme1/graph_spectra_all"

!python tools/theme1/analyze_graph_spectra.py \
    --outputs /content/pipeline/outputs \
    --spatial-models /content/pipeline/database/spatial_model \
    --output-dir {OUTPUT_DIR} \
    --weight {WEIGHT}

# Check results

In [ ]:
from pathlib import Path
import pandas as pd
import json

output_dir = Path(OUTPUT_DIR)

diagnostics = json.loads(
    (output_dir / "diagnostics.json").read_text(
        encoding="utf-8"
    )
)

print("=== 実行結果 ===")
print(json.dumps(
    diagnostics,
    ensure_ascii=False,
    indent=2,
))

energies = pd.read_csv(
    output_dir / "band_energies.csv"
)

signals = pd.read_csv(
    output_dir / "grain_signals.csv"
)

modes = pd.read_csv(
    output_dir / "mode_coefficients.csv"
)

print("\n=== 出力行数 ===")
print("band_energies:", len(energies))
print("grain_signals:", len(signals))
print("mode_coefficients:", len(modes))

# Save to Drive

In [ ]:
import shutil
from pathlib import Path
from datetime import datetime

source = Path(OUTPUT_DIR)

destination = (
    Path(DRIVE_PROJECT)
    / "results"
    / RESULT_FOLDER_NAME
    / "graph_spectra"
)

if destination.exists():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    destination = destination.with_name(
        f"graph_spectra_{timestamp}"
    )

shutil.copytree(source, destination)

print("全結果をDriveへ保存しました:")
print(destination)